<a href="https://colab.research.google.com/github/parshav42/learing/blob/main/Heart_disease_statlog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip -b /content/archive.zip -d /content


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as py
import seaborn as sn

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
df = pd.read_csv('/content/Heart_disease_statlog.csv')
df.head()

In [ ]:
X = df.iloc[:,:-1]
y=df.iloc[:,-1]

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test , y_train,y_test = train_test_split(X,y,test_size=0.2,shuffle=True)

In [ ]:
X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test = torch.tensor(X_test.values, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)


In [ ]:
class Model(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 10),
            nn.ReLU(),
            nn.Linear(10, 5),
            nn.ReLU(),
            nn.Linear(5, 3),
            nn.ReLU(),
            nn.Linear(3, 2),
            nn.ReLU(),
            nn.Linear(2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
model = Model(13)

# model.forward(X_train)
model = model.to(device)


loss_fun = nn.BCELoss()

optim = torch.optim.SGD(model.parameters(),lr=0.01,)


In [ ]:
epocs = 500
for epocs in range(epocs):



    y_prep = model(X_train.to(device))

    loss = loss_fun(y_prep,y_train.to(device))

    optim.zero_grad()

    loss.backward()

    optim.step()

    print(f"epocs{epocs +1} , loss = {loss}")




In [ ]:
model.eval()

In [ ]:
# Model evaluation
model.eval()

with torch.no_grad():
    y_pred = model(X_test.to(device))

    # Convert probabilities to 0/1
    y_pred_class = (y_pred >= 0.5).float()

    accuracy = (y_pred_class == y_test.to(device)).float().mean()

print("Test Accuracy:", accuracy.item())